<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/Dataset_Overview/DATASET_INSPECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Download directly to the Pro+ instance's local high-speed disk
!curl -L -o /content/massdata "https://www.dropbox.com/scl/fo/e9vabgqjq9ulf67ex595c/AEEXlTsl53kmgorOSJPnVGA?rlkey=7gsid4keqnxk7jsafl489w78z&st=i4wcvbwm&dl=0"

In [ ]:
# 1. Create the destination directory
!mkdir -p /content/data_extracted

# 2. Extract the Zip archive
!unzip /content/massdata -d /content/data_extracted

In [ ]:
import tarfile
import os

# Your paths
base_dir = '/content/data_extracted/MassData/extracted/synthea_1m_fhir_3_0_May_24'
tar_files = [os.path.join(base_dir, f) for f in os.listdir(base_dir) if f.endswith('.tar.gz')]

# Where all your final CSVs will live
master_csv_dir = '/content/Final_Thesis_Data'
os.makedirs(master_csv_dir, exist_ok=True)

print(f"Starting extraction of CSVs from {len(tar_files)} archives...")

for i, tar_path in enumerate(tar_files):
    print(f"Processing archive {i+1}/{len(tar_files)}: {os.path.basename(tar_path)}")

    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():
            # Only grab the CSVs
            if member.name.endswith('.csv'):
                # We extract it keeping the folder structure so files don't overwrite each other
                tar.extract(member, path=master_csv_dir)

print("\n✅ EXTRACTION COMPLETE!")
print(f"All your CSV files are now safely stored in: {master_csv_dir}")

Starting extraction of CSVs from 12 archives...
Processing archive 1/12: output_4_20170526T004637.tar.gz

/tmp/ipykernel_7643/3321792589.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=master_csv_dir)

Processing archive 2/12: output_9_20170527T185007.tar.gz
Processing archive 3/12: output_10_20170528T030916.tar.gz
Processing archive 4/12: output_3_20170525T161555.tar.gz
Processing archive 5/12: output_2_20170525T073836.tar.gz
Processing archive 6/12: output_5_20170526T091439.tar.gz
Processing archive 7/12: output_8_20170527T102552.tar.gz
Processing archive 8/12: output_11_20170528T113605.tar.gz
Processing archive 9/12: output_6_20170526T173337.tar.gz
Processing archive 10/12: output_1_20170524T232103.tar.gz
Processing archive 11/12: output_12_20170528T195303.tar.gz
Processing archive 12/12: output_7_20170527T015508.tar.gz

✅ EXTRACTION COMPLETE!
All your CSV files are now safely stored in: /content/Final_Thesis_Data


In [ ]:
# =============================================================================
# CELL: DATA INVENTORY & SCHEMA INSPECTION
# Purpose : Understand what CSV tables exist, their columns/types, and
#           how they link together via a shared patient identifier.
# Input   : /content/Final_Thesis_Data  (extracted CSV shards)
# Output  : Printed schema report — no files written
# =============================================================================

import glob
import os
import pandas as pd

RAW_DATA_DIR = '/content/Final_Thesis_Data'

# ── 1. Discover all CSV types ────────────────────────────────────────────────
all_csv_paths = glob.glob(f'{RAW_DATA_DIR}/**/*.csv', recursive=True)

# Group paths by table name (e.g. "conditions.csv"), keep one example per type
table_index = {}
for path in all_csv_paths:
    name = os.path.basename(path)
    if name not in table_index:
        table_index[name] = path

print(f"Found {len(table_index)} unique table types across "
      f"{len(all_csv_paths)} total CSV shards.\n")
print("=" * 65)

# ── 2. Schema report for each table ──────────────────────────────────────────
shared_id_col = None   # will capture the patient link column name

for table_name, sample_path in sorted(table_index.items()):
    df = pd.read_csv(sample_path, nrows=3, low_memory=False)

    print(f"\n📄  {table_name.upper()}")
    print(f"    Sample file : {os.path.relpath(sample_path, RAW_DATA_DIR)}")
    print(f"    Columns ({len(df.columns)}):")

    for col in df.columns:
        dtype   = df[col].dtype
        example = df[col].dropna().iloc[0] if not df[col].dropna().empty else "—"
        # Flag candidate patient-link columns
        flag = " ◀ patient link?" if col.upper() in ('PATIENT', 'ID', 'PATIENTID') else ""
        print(f"      {col:<30}  {str(dtype):<12}  e.g. {str(example)[:40]}{flag}")

    # Track link column
    for col in df.columns:
        if col.upper() == 'PATIENT':
            shared_id_col = 'PATIENT'
        elif col.upper() == 'ID' and table_name == 'patients.csv':
            shared_id_col = 'ID'  # patients table uses ID not PATIENT

print("\n" + "=" * 65)

# ── 3. Relationship summary ───────────────────────────────────────────────────
print("\n🔗  TABLE RELATIONSHIPS")
print(f"    patients.csv  →  ID  column  (primary key — one row per patient)")
print(f"    all others    →  PATIENT column  (foreign key → patients.ID)\n")

print("    Table                     | Patient link column")
print("    " + "-" * 50)
for table_name, sample_path in sorted(table_index.items()):
    df_cols = pd.read_csv(sample_path, nrows=0).columns.tolist()
    if 'PATIENT' in df_cols:
        link = 'PATIENT  →  patients.ID'
    elif 'ID' in df_cols and table_name == 'patients.csv':
        link = 'ID  (primary key)'
    else:
        link = 'no direct patient link found'
    print(f"    {table_name:<28}| {link}")

# ── 4. Quick row-count estimate (first shard only, multiply × 12 to estimate) ─
print("\n📊  ROW COUNT ESTIMATE  (first shard × 12 ≈ full dataset)")
print(f"    {'Table':<28}  {'Rows in shard 1':>16}  {'Est. total':>12}")
print("    " + "-" * 58)
for table_name, sample_path in sorted(table_index.items()):
    n = sum(1 for _ in open(sample_path)) - 1   # fast line count
    print(f"    {table_name:<28}  {n:>16,}  {n*12:>12,}")

In [ ]:
Found 9 unique table types across 108 total CSV shards.

=================================================================

📄  ALLERGIES.CSV
    Sample file : output_7/csv/allergies.csv
    Columns (6):
      START                           object        e.g. 2010-11-09
      STOP                            float64       e.g. —
      PATIENT                         object        e.g. 1e64ebab-b785-40ff-82cb-c0a1449c5d93 ◀ patient link?
      ENCOUNTER                       object        e.g. 8ffb56d8-078b-42e5-ab50-e592b31e8185
      CODE                            int64         e.g. 91935009
      DESCRIPTION                     object        e.g. Allergy to peanuts

📄  CAREPLANS.CSV
    Sample file : output_7/csv/careplans.csv
    Columns (9):
      ID                              object        e.g. 9ec2edf5-a460-4680-b697-fa7e05b5534a ◀ patient link?
      START                           object        e.g. 2011-03-01
      STOP                            object        e.g. 2011-05-13
      PATIENT                         object        e.g. a60f5f2f-cf37-4952-b127-8f3ddd3fc880 ◀ patient link?
      ENCOUNTER                       object        e.g. 69756436-c40b-427e-9c19-0914f8a23ba9
      CODE                            int64         e.g. 385691007
      DESCRIPTION                     object        e.g. Fracture care
      REASONCODE                      int64         e.g. 58150001
      REASONDESCRIPTION               object        e.g. Fracture of clavicle

📄  CONDITIONS.CSV
    Sample file : output_7/csv/conditions.csv
    Columns (6):
      START                           object        e.g. 2014-01-31
      STOP                            object        e.g. 2014-03-06
      PATIENT                         object        e.g. 6ea6a21c-2123-47cb-9e23-8776cf155ea0 ◀ patient link?
      ENCOUNTER                       object        e.g. 89f51831-7eb0-4494-a664-af535a5ac798
      CODE                            int64         e.g. 65363002
      DESCRIPTION                     object        e.g. Otitis media

📄  ENCOUNTERS.CSV
    Sample file : output_7/csv/encounters.csv
    Columns (7):
      ID                              object        e.g. da595b6c-8a8e-4cb2-bb07-4e55bea8a973 ◀ patient link?
      DATE                            object        e.g. 2014-06-14
      PATIENT                         object        e.g. 863ba65e-b907-4856-9896-d175ba3b22a1 ◀ patient link?
      CODE                            int64         e.g. 170258001
      DESCRIPTION                     object        e.g. Outpatient Encounter
      REASONCODE                      float64       e.g. —
      REASONDESCRIPTION               float64       e.g. —

📄  IMMUNIZATIONS.CSV
    Sample file : output_7/csv/immunizations.csv
    Columns (5):
      DATE                            object        e.g. 2014-06-14
      PATIENT                         object        e.g. 863ba65e-b907-4856-9896-d175ba3b22a1 ◀ patient link?
      ENCOUNTER                       object        e.g. da595b6c-8a8e-4cb2-bb07-4e55bea8a973
      CODE                            int64         e.g. 8
      DESCRIPTION                     object        e.g. Hep B  adolescent or pediatric

📄  MEDICATIONS.CSV
    Sample file : output_7/csv/medications.csv
    Columns (8):
      START                           object        e.g. 2014-01-31
      STOP                            object        e.g. 2014-02-14
      PATIENT                         object        e.g. 6ea6a21c-2123-47cb-9e23-8776cf155ea0 ◀ patient link?
      ENCOUNTER                       object        e.g. 89f51831-7eb0-4494-a664-af535a5ac798
      CODE                            int64         e.g. 392151
      DESCRIPTION                     object        e.g. Amoxicillin 200 MG Oral Tablet
      REASONCODE                      float64       e.g. —
      REASONDESCRIPTION               float64       e.g. —

📄  OBSERVATIONS.CSV
    Sample file : output_7/csv/observations.csv
    Columns (7):
      DATE                            object        e.g. 2014-07-21
      PATIENT                         object        e.g. 863ba65e-b907-4856-9896-d175ba3b22a1 ◀ patient link?
      ENCOUNTER                       object        e.g. 82e91792-9c8a-4463-8382-40b74b543a35
      CODE                            object        e.g. 8302-2
      DESCRIPTION                     object        e.g. Body Height
      VALUE                           float64       e.g. 57.67
      UNITS                           object        e.g. cm

📄  PATIENTS.CSV
    Sample file : output_7/csv/patients.csv
    Columns (17):
      ID                              object        e.g. 863ba65e-b907-4856-9896-d175ba3b22a1 ◀ patient link?
      BIRTHDATE                       object        e.g. 2014-06-14
      DEATHDATE                       float64       e.g. —
      SSN                             object        e.g. 999-98-4651
      DRIVERS                         object        e.g. S99911641
      PASSPORT                        float64       e.g. —
      PREFIX                          object        e.g. Ms.
      FIRST                           object        e.g. Hanna390
      LAST                            object        e.g. Auer342
      SUFFIX                          float64       e.g. —
      MAIDEN                          float64       e.g. —
      MARITAL                         float64       e.g. —
      RACE                            object        e.g. white
      ETHNICITY                       object        e.g. italian
      GENDER                          object        e.g. M
      BIRTHPLACE                      object        e.g. Sharon MA US
      ADDRESS                         object        e.g. 2994 Pinkie Spring Bourne MA 02559 US

📄  PROCEDURES.CSV
    Sample file : output_7/csv/procedures.csv
    Columns (7):
      DATE                            object        e.g. 2014-08-25
      PATIENT                         object        e.g. 863ba65e-b907-4856-9896-d175ba3b22a1 ◀ patient link?
      ENCOUNTER                       object        e.g. 1ae51bf3-626a-4f4c-a962-af7c137d2b92
      CODE                            int64         e.g. 428191000124101
      DESCRIPTION                     object        e.g. Documentation of current medications
      REASONCODE                      float64       e.g. —
      REASONDESCRIPTION               float64       e.g. —

=================================================================

🔗  TABLE RELATIONSHIPS
    patients.csv  →  ID  column  (primary key — one row per patient)
    all others    →  PATIENT column  (foreign key → patients.ID)

    Table                     | Patient link column
    --------------------------------------------------
    allergies.csv               | PATIENT  →  patients.ID
    careplans.csv               | PATIENT  →  patients.ID
    conditions.csv              | PATIENT  →  patients.ID
    encounters.csv              | PATIENT  →  patients.ID
    immunizations.csv           | PATIENT  →  patients.ID
    medications.csv             | PATIENT  →  patients.ID
    observations.csv            | PATIENT  →  patients.ID
    patients.csv                | ID  (primary key)
    procedures.csv              | PATIENT  →  patients.ID

📊  ROW COUNT ESTIMATE  (first shard × 12 ≈ full dataset)
    Table                          Rows in shard 1    Est. total
    ----------------------------------------------------------
    allergies.csv                           52,357       628,284
    careplans.csv                          792,513     9,510,156
    conditions.csv                         482,052     5,784,624
    encounters.csv                       1,255,619    15,067,428
    immunizations.csv                      868,143    10,417,716
    medications.csv                        394,937     4,739,244
    observations.csv                     5,372,043    64,464,516
    patients.csv                           132,721     1,592,652
    procedures.csv                         622,581     7,470,972


In [ ]:
# =============================================================================
# CELL: MISSING DATA AUDIT
# Purpose : Report % missing values per column, for every table type.
#           Uses ALL 12 shards (not just one) for accuracy.
# Input   : /content/Final_Thesis_Data  (extracted CSV shards)
# Output  : Printed missing data report — no files written
# =============================================================================

import glob
import os
import pandas as pd

RAW_DATA_DIR = '/content/Final_Thesis_Data'

all_csv_paths = glob.glob(f'{RAW_DATA_DIR}/**/*.csv', recursive=True)

# Group ALL shards by table name
table_shards = {}
for path in all_csv_paths:
    name = os.path.basename(path)
    table_shards.setdefault(name, []).append(path)

print("MISSING DATA REPORT — across all 12 shards")
print("=" * 65)

for table_name, shard_paths in sorted(table_shards.items()):

    # Load all shards for this table type and concatenate
    chunks = []
    for path in shard_paths:
        try:
            chunks.append(pd.read_csv(path, low_memory=False))
        except Exception as e:
            print(f"  ⚠️  Could not read {path}: {e}")

    if not chunks:
        continue

    df = pd.concat(chunks, ignore_index=True)
    total_rows = len(df)

    print(f"\n📄  {table_name.upper()}  ({total_rows:,} total rows)")
    print(f"    {'Column':<30}  {'Missing N':>10}  {'Missing %':>10}")
    print("    " + "-" * 54)

    for col in df.columns:
        n_missing = df[col].isna().sum()
        pct       = 100 * n_missing / total_rows if total_rows > 0 else 0
        # Highlight columns with >5% missing
        flag = " ⚠️" if pct > 5 else ""
        print(f"    {col:<30}  {n_missing:>10,}  {pct:>9.1f}%{flag}")

print("\n" + "=" * 65)
print("⚠️  = column has more than 5% missing values")

In [ ]:
MISSING DATA REPORT — across all 12 shards
=================================================================

📄  ALLERGIES.CSV  (624,611 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    START                                    0        0.0%
    STOP                               582,328       93.2% ⚠️
    PATIENT                                  0        0.0%
    ENCOUNTER                                0        0.0%
    CODE                                     0        0.0%
    DESCRIPTION                              0        0.0%

📄  CAREPLANS.CSV  (9,558,659 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    ID                                       0        0.0%
    START                                    0        0.0%
    STOP                             5,011,582       52.4% ⚠️
    PATIENT                                  0        0.0%
    ENCOUNTER                                0        0.0%
    CODE                                     0        0.0%
    DESCRIPTION                              0        0.0%
    REASONCODE                       1,133,908       11.9% ⚠️
    REASONDESCRIPTION                1,133,908       11.9% ⚠️

📄  CONDITIONS.CSV  (5,809,954 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    START                                    0        0.0%
    STOP                             2,619,842       45.1% ⚠️
    PATIENT                                  0        0.0%
    ENCOUNTER                                0        0.0%
    CODE                                     0        0.0%
    DESCRIPTION                              0        0.0%

📄  ENCOUNTERS.CSV  (15,109,427 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    ID                                       0        0.0%
    DATE                                     0        0.0%
    PATIENT                                  0        0.0%
    CODE                                     0        0.0%
    DESCRIPTION                              0        0.0%
    REASONCODE                      10,847,396       71.8% ⚠️
    REASONDESCRIPTION               10,847,396       71.8% ⚠️

📄  IMMUNIZATIONS.CSV  (10,412,118 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    DATE                                     0        0.0%
    PATIENT                                  0        0.0%
    ENCOUNTER                                0        0.0%
    CODE                                     0        0.0%
    DESCRIPTION                              0        0.0%

📄  MEDICATIONS.CSV  (4,781,956 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    START                                    0        0.0%
    STOP                             2,601,594       54.4% ⚠️
    PATIENT                                  0        0.0%
    ENCOUNTER                                0        0.0%
    CODE                                     0        0.0%
    DESCRIPTION                              0        0.0%
    REASONCODE                       1,418,598       29.7% ⚠️
    REASONDESCRIPTION                1,418,598       29.7% ⚠️

📄  OBSERVATIONS.CSV  (64,654,706 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    DATE                                     0        0.0%
    PATIENT                                  0        0.0%
    ENCOUNTER                                0        0.0%
    CODE                                     0        0.0%
    DESCRIPTION                              0        0.0%
    VALUE                              169,257        0.3%
    UNITS                              169,257        0.3%
  ⚠️  Could not read /content/Final_Thesis_Data/output_7/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 1294, saw 18

  ⚠️  Could not read /content/Final_Thesis_Data/output_2/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 1189, saw 32

  ⚠️  Could not read /content/Final_Thesis_Data/output_8/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 9, saw 32

  ⚠️  Could not read /content/Final_Thesis_Data/output_10/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 142, saw 31

  ⚠️  Could not read /content/Final_Thesis_Data/output_3/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 1265, saw 40

  ⚠️  Could not read /content/Final_Thesis_Data/output_11/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 327, saw 18

  ⚠️  Could not read /content/Final_Thesis_Data/output_6/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 439, saw 23

  ⚠️  Could not read /content/Final_Thesis_Data/output_5/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 310, saw 30

  ⚠️  Could not read /content/Final_Thesis_Data/output_12/csv/patients.csv: Error tokenizing data. C error: Expected 17 fields in line 3, saw 32

  ⚠️  Could not read /content/Final_Thesis_Data/output_9/csv/patients.csv: Error tokenizing data. C error: Expected 33 fields in line 53, saw 39


📄  PATIENTS.CSV  (265,724 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    ID                                     371        0.1%
    BIRTHDATE                          265,504       99.9% ⚠️
    DEATHDATE                          265,546       99.9% ⚠️
    SSN                                265,530       99.9% ⚠️
    DRIVERS                            265,555       99.9% ⚠️
    PASSPORT                           265,560       99.9% ⚠️
    PREFIX                             265,576       99.9% ⚠️
    FIRST                              265,568       99.9% ⚠️
    LAST                               265,578       99.9% ⚠️
    SUFFIX                             265,621      100.0% ⚠️
    MAIDEN                             265,607      100.0% ⚠️
    MARITAL                            265,582       99.9% ⚠️
    RACE                               265,569       99.9% ⚠️
    ETHNICITY                          265,587       99.9% ⚠️
    GENDER                             265,593      100.0% ⚠️
    BIRTHPLACE                         265,623      100.0% ⚠️
    ADDRESS                            265,624      100.0% ⚠️

📄  PROCEDURES.CSV  (7,502,018 total rows)
    Column                           Missing N   Missing %
    ------------------------------------------------------
    DATE                                     0        0.0%
    PATIENT                                  0        0.0%
    ENCOUNTER                                0        0.0%
    CODE                                     0        0.0%
    DESCRIPTION                              0        0.0%
    REASONCODE                       5,604,154       74.7% ⚠️
    REASONDESCRIPTION                5,604,154       74.7% ⚠️

=================================================================
⚠️  = column has more than 5% missing values


In [ ]:
# =============================================================================
# CELL: PATIENTS TABLE — MISSING DATA AUDIT (ADDRESS column skipped)
# Purpose : patients.csv has unquoted commas in ADDRESS which breaks the
#           CSV parser. We simply skip that column as it is not needed.
# Input   : /content/Final_Thesis_Data  (all 12 shards)
# Output  : df_patients loaded in memory + printed missing data report
# =============================================================================

import glob
import pandas as pd

RAW_DATA_DIR = '/content/Final_Thesis_Data'

patient_shards = sorted(glob.glob(f'{RAW_DATA_DIR}/**/patients.csv', recursive=True))
print(f"Found {len(patient_shards)} patients.csv shards.\n")

# ADDRESS is excluded — it contains unquoted commas that break the parser
PATIENT_COLS = [
    'ID', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT',
    'PREFIX', 'FIRST', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL',
    'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE'
]

chunks = []

for path in patient_shards:
    try:
        df = pd.read_csv(path, usecols=PATIENT_COLS, low_memory=False)
        chunks.append(df)
        print(f"  ✅  {path.split('/')[-3]}  →  {len(df):,} rows loaded")
    except Exception as e:
        print(f"  ❌  {path.split('/')[-3]}  →  {e}")

df_patients = pd.concat(chunks, ignore_index=True)
total_rows  = len(df_patients)

print(f"\n📄  PATIENTS.CSV  —  {total_rows:,} total rows across all shards")
print(f"    {'Column':<30}  {'Missing N':>10}  {'Missing %':>10}")
print("    " + "-" * 54)

for col in df_patients.columns:
    n_missing = df_patients[col].isna().sum()
    pct       = 100 * n_missing / total_rows
    flag      = " ⚠️" if pct > 5 else ""
    print(f"    {col:<30}  {n_missing:>10,}  {pct:>9.1f}%{flag}")

In [ ]:
Found 12 patients.csv shards.

  ✅  output_1  →  132,896 rows loaded
  ✅  output_10  →  132,094 rows loaded
  ✅  output_11  →  132,540 rows loaded
  ✅  output_12  →  133,076 rows loaded
  ✅  output_2  →  133,420 rows loaded
  ✅  output_3  →  132,681 rows loaded
  ✅  output_4  →  132,828 rows loaded
  ✅  output_5  →  133,443 rows loaded
  ✅  output_6  →  132,801 rows loaded
  ✅  output_7  →  132,691 rows loaded
  ✅  output_8  →  133,040 rows loaded
  ✅  output_9  →  133,201 rows loaded

📄  PATIENTS.CSV  —  1,594,711 total rows across all shards
    Column                           Missing N   Missing %
    ------------------------------------------------------
    ID                                     901        0.1%
    BIRTHDATE                          399,078       25.0% ⚠️
    DEATHDATE                        1,295,318       81.2% ⚠️
    SSN                                399,934       25.1% ⚠️
    DRIVERS                            580,576       36.4% ⚠️
    PASSPORT                           633,865       39.7% ⚠️
    PREFIX                             607,249       38.1% ⚠️
    FIRST                              399,976       25.1% ⚠️
    LAST                               399,905       25.1% ⚠️
    SUFFIX                           1,580,343       99.1% ⚠️
    MAIDEN                           1,252,801       78.6% ⚠️
    MARITAL                            735,807       46.1% ⚠️
    RACE                               400,004       25.1% ⚠️
    ETHNICITY                          399,958       25.1% ⚠️
    GENDER                             400,235       25.1% ⚠️
    BIRTHPLACE                         400,249       25.1% ⚠️


In [ ]:
# =============================================================================
# CELL: COHORT BUILDING & ATTRITION FLOWCHART
# Purpose : Starting from the full 1M dataset, this cell computes:
#           (1) Total patients in dataset
#           (2) Observational period overall + per index date
#           (3) Total T2D patients (diagnosis codes)
#           (4) T2D patients validated as Type 2 specifically
#           (5) T2D patients with at least one treatment
#           (6) Eligible patients per index date (Td, T0, T0+5, T0+10)
# Input   : /content/Final_Thesis_Data  (all 12 raw CSV shards)
# Output  : cohort DataFrame in memory
#           /content/cohort_base.csv
# Packages: pandas, glob, numpy
# =============================================================================

import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_DATA_DIR = '/content/Final_Thesis_Data'
STUDY_END    = pd.Timestamp('2017-05-28')

# ── Code lists ────────────────────────────────────────────────────────────────
# All codes that can indicate a T2D diagnosis
T2D_CODES = [
    44054006,        # Diabetes (generic — needs validation below)
    127013003,       # Diabetic renal disease
    60951000119105,  # Blindness due to T2D
    422034002,       # Diabetic retinopathy
    97331000119101,  # Macular edema due to T2D
    90781000119102,  # Microalbuminuria due to T2D
    368581000119106, # Neuropathy due to T2D
    1551000119108,   # NPDR due to T2D
    1501000119109,   # Proliferative DR due to T2D
    157141000119108, # Proteinuria due to T2D
]

# Codes that CONFIRM Type 2 specifically (excludes generic 44054006)
T2D_SPECIFIC_CODES = [
    60951000119105, 422034002,  97331000119101,
    90781000119102, 368581000119106, 1551000119108,
    1501000119109,  157141000119108,
    127013003,      # diabetic renal disease also specific to T2D
]

# RxNorm codes for T2D medications in Synthea
T2D_MED_CODES = [
    860975,   # Metformin
    1373463,  # Canagliflozin (SGLT2i)
    897122,   # Liraglutide (GLP-1)
    106892,   # Insulin (Humalog / Humulin)
]

# Tables that carry diagnosis codes
DIAG_TABLES = {
    'conditions.csv': {'date_col': 'START', 'code_col': 'CODE'},
    'encounters.csv': {'date_col': 'DATE',  'code_col': 'REASONCODE'},
    'careplans.csv':  {'date_col': 'START', 'code_col': 'REASONCODE'},
    'procedures.csv': {'date_col': 'DATE',  'code_col': 'REASONCODE'},
}

print("=" * 65)
print("COHORT BUILDING — ATTRITION FLOWCHART")
print("=" * 65)

# ── STEP 1 : Total patients ───────────────────────────────────────────────────
print("\nSTEP 1 — Counting total patients...")

pat_files  = sorted(glob.glob(f'{RAW_DATA_DIR}/**/patients.csv', recursive=True))
pat_chunks = []
for f in pat_files:
    try:
        pat_chunks.append(pd.read_csv(
            f, usecols=['ID', 'BIRTHDATE', 'DEATHDATE'], low_memory=False))
    except Exception:
        pass

df_patients = (pd.concat(pat_chunks, ignore_index=True)
                 .rename(columns={'ID': 'PATIENT'}))
df_patients['BIRTHDATE'] = pd.to_datetime(df_patients['BIRTHDATE'], errors='coerce')
df_patients['DEATHDATE'] = pd.to_datetime(df_patients['DEATHDATE'], errors='coerce')
df_patients = df_patients.dropna(subset=['BIRTHDATE']).drop_duplicates('PATIENT')

N_TOTAL = len(df_patients)
print(f"  Total patients : {N_TOTAL:,}")

# ── STEP 2 : Overall observational period ────────────────────────────────────
print("\nSTEP 2 — Overall observational period...")

DATE_TABLES = {
    'conditions.csv':   'START',
    'medications.csv':  'START',
    'encounters.csv':   'DATE',
    'observations.csv': 'DATE',
    'procedures.csv':   'DATE',
}

all_dates = []
for table, date_col in DATE_TABLES.items():
    for f in glob.glob(f'{RAW_DATA_DIR}/**/{table}', recursive=True):
        try:
            chunk = pd.read_csv(f, usecols=[date_col], low_memory=False)
            all_dates.append(pd.to_datetime(chunk[date_col], errors='coerce').dropna())
        except Exception:
            pass

all_dates_series = pd.concat(all_dates, ignore_index=True)
OBS_START = all_dates_series.min()
OBS_END   = all_dates_series.max()
print(f"  First recorded event : {OBS_START.date()}")
print(f"  Last  recorded event : {OBS_END.date()}")
print(f"  Total span           : {(OBS_END - OBS_START).days / 365.25:.1f} years")

# ── STEP 3 : T2D patients — first diagnosis date (Td) ────────────────────────
print("\nSTEP 3 — Identifying T2D patients and diagnosis date (Td)...")

td_chunks = []
for table, cols in DIAG_TABLES.items():
    for f in glob.glob(f'{RAW_DATA_DIR}/**/{table}', recursive=True):
        try:
            chunk = pd.read_csv(
                f, usecols=['PATIENT', cols['date_col'], cols['code_col']],
                low_memory=False)
            chunk[cols['code_col']] = pd.to_numeric(
                chunk[cols['code_col']], errors='coerce')
            rows = chunk[chunk[cols['code_col']].isin(T2D_CODES)].copy()
            if not rows.empty:
                rows = rows.rename(columns={cols['date_col']: 'Td'})
                td_chunks.append(rows[['PATIENT', 'Td']])
        except Exception:
            pass

df_td = (pd.concat(td_chunks, ignore_index=True)
           .assign(Td=lambda x: pd.to_datetime(x['Td'], errors='coerce'))
           .dropna(subset=['Td'])
           .groupby('PATIENT')['Td'].min()
           .reset_index())

N_T2D_ANY = len(df_td)
print(f"  T2D patients (any code) : {N_T2D_ANY:,}  "
      f"({100*N_T2D_ANY/N_TOTAL:.1f}% of dataset)")

# ── STEP 4 : Validate as Type 2 specifically ─────────────────────────────────
# Patients with ONLY generic code 44054006 need at least one T2D-specific
# complication code to be confirmed as Type 2 (excludes T1D, gestational, etc.)
print("\nSTEP 4 — Validating as Type 2 specifically...")

# Who only has the generic code?
cond_files = glob.glob(f'{RAW_DATA_DIR}/**/conditions.csv', recursive=True)
all_t2d_ids    = set(df_td['PATIENT'])
specific_ids   = set()   # patients who have at least one T2D-specific code

for f in cond_files:
    try:
        chunk = pd.read_csv(f, usecols=['PATIENT', 'CODE'], low_memory=False)
        chunk = chunk[chunk['PATIENT'].isin(all_t2d_ids)]
        chunk['CODE'] = pd.to_numeric(chunk['CODE'], errors='coerce')
        matches = chunk[chunk['CODE'].isin(T2D_SPECIFIC_CODES)]
        specific_ids.update(matches['PATIENT'].unique())
    except Exception:
        pass

# Keep patient if they have a specific code OR their diagnosis code
# was already specific (not just 44054006)
validated_ids = specific_ids   # all patients with at least one specific code
df_td_validated = df_td[df_td['PATIENT'].isin(validated_ids)].copy()

N_T2D_VALIDATED = len(df_td_validated)
print(f"  Validated T2D patients  : {N_T2D_VALIDATED:,}  "
      f"({100*N_T2D_VALIDATED/N_T2D_ANY:.1f}% of T2D any-code patients)")

# ── STEP 5 : First treatment date (T0) ───────────────────────────────────────
print("\nSTEP 5 — Finding first treatment date (T0)...")

t0_chunks = []
for f in glob.glob(f'{RAW_DATA_DIR}/**/medications.csv', recursive=True):
    try:
        chunk = pd.read_csv(f, usecols=['PATIENT', 'CODE', 'START'], low_memory=False)
        chunk['CODE'] = pd.to_numeric(chunk['CODE'], errors='coerce')
        rows = chunk[chunk['CODE'].isin(T2D_MED_CODES)].copy()
        if not rows.empty:
            t0_chunks.append(rows[['PATIENT', 'START']])
    except Exception:
        pass

df_t0 = (pd.concat(t0_chunks, ignore_index=True)
           .assign(T0=lambda x: pd.to_datetime(x['START'], errors='coerce'))
           .dropna(subset=['T0'])
           .groupby('PATIENT')['T0'].min()
           .reset_index()[['PATIENT', 'T0']])

# Merge everything — only validated T2D patients
cohort = (df_td_validated
          .merge(df_t0, on='PATIENT', how='left')
          .merge(df_patients[['PATIENT', 'BIRTHDATE', 'DEATHDATE']],
                 on='PATIENT', how='inner'))

N_T2D_TREATED = cohort['T0'].notna().sum()
print(f"  Validated T2D + treatment : {N_T2D_TREATED:,}  "
      f"({100*N_T2D_TREATED/N_T2D_VALIDATED:.1f}% of validated T2D)")

# ── STEP 6 : Eligibility per index date ──────────────────────────────────────
print("\nSTEP 6 — Applying eligibility criteria per index date...")

cohort = cohort.dropna(subset=['T0']).copy()
cohort['Observation_End'] = cohort['DEATHDATE'].fillna(STUDY_END)
cohort['Is_Dead']         = cohort['DEATHDATE'].notna()

# Index dates
cohort['T0_plus_5']  = cohort['T0'] + pd.DateOffset(years=5)
cohort['T0_plus_10'] = cohort['T0'] + pd.DateOffset(years=10)

def is_eligible(df, index_col):
    idx         = df[index_col]
    lookback_ok = (idx - df['BIRTHDATE']).dt.days / 365.25 >= 5
    followup_ok = (
        ((df['Observation_End'] - idx).dt.days / 365.25 >= 5) |
        (df['Is_Dead'] & (df['Observation_End'] >= idx))
    )
    return lookback_ok & followup_ok

def obs_period(df, index_col, eligible_mask):
    sub = df[eligible_mask]
    idx = sub[index_col]
    fu_end = sub[['Observation_End',
                  (idx + pd.DateOffset(years=5)).rename('cap')]
                 ].min(axis=1) if False else \
             pd.concat([sub['Observation_End'],
                        idx + pd.DateOffset(years=5)], axis=1).min(axis=1)
    return idx.min().date(), fu_end.max().date(), \
           round((fu_end - idx).dt.days.median() / 365.25, 1)

cohort['Elig_Td']        = is_eligible(cohort, 'Td')
cohort['Elig_T0']        = is_eligible(cohort, 'T0')
cohort['Elig_T0_plus5']  = is_eligible(cohort, 'T0_plus_5')
cohort['Elig_T0_plus10'] = is_eligible(cohort, 'T0_plus_10')

# ── FINAL REPORT ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("ATTRITION FLOWCHART")
print("=" * 65)
print(f"  Full dataset                              : {N_TOTAL:>8,}")
print(f"  Observational period (dataset)            : "
      f"{OBS_START.date()} → {OBS_END.date()}")
print(f"  ↓ T2D diagnosis (any code)                : {N_T2D_ANY:>8,}")
print(f"  ↓ Validated Type 2 specifically           : {N_T2D_VALIDATED:>8,}")
print(f"  ↓ Validated T2D + at least one treatment  : {N_T2D_TREATED:>8,}")

print(f"\n  {'Index date':<15} {'N eligible':>10}   {'Period':<32}  {'Median FU'}")
print("  " + "-" * 70)

for label, idx_col, elig_col in [
    ('Td  (diagnosis)',  'Td',         'Elig_Td'),
    ('T0  (treatment)',  'T0',         'Elig_T0'),
    ('T0+5  (5-year)',   'T0_plus_5',  'Elig_T0_plus5'),
    ('T0+10 (10-year)',  'T0_plus_10', 'Elig_T0_plus10'),
]:
    mask   = cohort[elig_col]
    n      = mask.sum()
    sub    = cohort[mask]
    idx    = sub[idx_col]
    fu_end = pd.concat([sub['Observation_End'],
                        idx + pd.DateOffset(years=5)], axis=1).min(axis=1)
    period = f"{idx.min().date()} → {fu_end.max().date()}"
    med_fu = f"{(fu_end - idx).dt.days.median() / 365.25:.1f} yrs"
    print(f"  {label:<15} {n:>10,}   {period:<32}  {med_fu}")

print("=" * 65)

cohort.to_csv('/content/cohort_base.csv', index=False)
print("\n✅ Saved to /content/cohort_base.csv")

In [ ]:


=================================================================
COHORT BUILDING — ATTRITION FLOWCHART
=================================================================

STEP 1 — Counting total patients...
  Total patients : 1,193,623

STEP 2 — Overall observational period...
  First recorded event : 1906-08-08
  Last  recorded event : 2017-05-30
  Total span           : 110.8 years

STEP 3 — Identifying T2D patients and diagnosis date (Td)...
  T2D patients (any code) : 86,604  (7.3% of dataset)

STEP 4 — Validating as Type 2 specifically...
  Validated T2D patients  : 62,204  (71.8% of T2D any-code patients)

STEP 5 — Finding first treatment date (T0)...
  Validated T2D + treatment : 44,714  (71.9% of validated T2D)

STEP 6 — Applying eligibility criteria per index date...

=================================================================
ATTRITION FLOWCHART
=================================================================
  Full dataset                              : 1,193,623
  Observational period (dataset)            : 1906-08-08 → 2017-05-30
  ↓ T2D diagnosis (any code)                :   86,604
  ↓ Validated Type 2 specifically           :   62,204
  ↓ Validated T2D + at least one treatment  :   44,714

  Index date      N eligible   Period                            Median FU
  ----------------------------------------------------------------------
  Td  (diagnosis)     43,774   1927-10-23 → 2017-05-27           5.0 yrs
  T0  (treatment)     42,504   1931-08-28 → 2017-05-27           5.0 yrs
  T0+5  (5-year)      38,169   1936-08-28 → 2017-05-27           5.0 yrs
  T0+10 (10-year)     32,130   1941-08-28 → 2017-05-27           5.0 yrs
=================================================================

✅ Saved to /content/cohort_base.csv


In [ ]:
# =============================================================================
# CELL 1: MASTER COHORT CSV  (Type 2 validated, correct attrition)
# Purpose : Build one clean CSV with all patient-level information.
#           Validation logic: a patient is confirmed T2D only if they have
#           at least one T2D-SPECIFIC complication code (not just the
#           generic 44054006 "Diabetes" which could be T1D or gestational).
#           Patients with ONLY 44054006 are dropped unless they also have
#           a T2D-specific complication code.
# Input   : /content/Final_Thesis_Data  (all 12 raw CSV shards)
# Output  : /content/cohort_master.csv
# Packages: pandas, glob, numpy
# =============================================================================

import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_DATA_DIR = '/content/Final_Thesis_Data'
STUDY_END    = pd.Timestamp('2017-05-28')

# ── Code lists ────────────────────────────────────────────────────────────────
# Generic diabetes code — could be T1D, T2D, or gestational
GENERIC_DIABETES_CODE = 44054006

# T2D-SPECIFIC codes: these only appear in Type 2 patients
# Using any of these = confirmed T2D, regardless of whether 44054006 is present
T2D_SPECIFIC_CODES = [
    127013003,       # Diabetic renal disease
    60951000119105,  # Blindness due to T2D
    422034002,       # Diabetic retinopathy due to T2D
    97331000119101,  # Macular edema due to T2D
    90781000119102,  # Microalbuminuria due to T2D
    368581000119106, # Neuropathy due to T2D
    1551000119108,   # NPDR due to T2D
    1501000119109,   # Proliferative DR due to T2D
    157141000119108, # Proteinuria due to T2D
]

# All codes used to find the initial T2D population (cast net wide first)
T2D_ALL_CODES = [GENERIC_DIABETES_CODE] + T2D_SPECIFIC_CODES

# RxNorm codes for T2D medications in Synthea
T2D_MED_CODES = {
    860975:  'Metformin',
    1373463: 'Canagliflozin (SGLT2i)',
    897122:  'Liraglutide (GLP-1)',
    106892:  'Insulin',
}

# Tables that can carry a diagnosis code
DIAG_TABLES = {
    'conditions.csv': {'date_col': 'START', 'code_col': 'CODE',
                       'desc_col': 'DESCRIPTION'},
    'encounters.csv': {'date_col': 'DATE',  'code_col': 'REASONCODE',
                       'desc_col': 'REASONDESCRIPTION'},
    'careplans.csv':  {'date_col': 'START', 'code_col': 'REASONCODE',
                       'desc_col': 'REASONDESCRIPTION'},
    'procedures.csv': {'date_col': 'DATE',  'code_col': 'REASONCODE',
                       'desc_col': 'REASONDESCRIPTION'},
}

print("=" * 65)
print("COHORT BUILDING")
print("=" * 65)

# ── STEP 1 : Patient demographics ────────────────────────────────────────────
print("\nSTEP 1 — Loading patient demographics...")
pat_chunks = []
for f in sorted(glob.glob(f'{RAW_DATA_DIR}/**/patients.csv', recursive=True)):
    try:
        pat_chunks.append(pd.read_csv(
            f, usecols=['ID','BIRTHDATE','DEATHDATE','GENDER','RACE','ETHNICITY'],
            low_memory=False))
    except Exception:
        pass

df_patients = (pd.concat(pat_chunks, ignore_index=True)
                 .rename(columns={'ID': 'PATIENT'})
                 .drop_duplicates('PATIENT'))
df_patients['BIRTHDATE'] = pd.to_datetime(df_patients['BIRTHDATE'], errors='coerce')
df_patients['DEATHDATE'] = pd.to_datetime(df_patients['DEATHDATE'], errors='coerce')
df_patients = df_patients.dropna(subset=['BIRTHDATE'])
print(f"  {len(df_patients):,} patients loaded")

# ── STEP 2 : Find all T2D-coded patients + their earliest diagnosis date ──────
print("\nSTEP 2 — Finding T2D diagnosis dates (Td)...")
td_chunks = []
for table, cols in DIAG_TABLES.items():
    for f in glob.glob(f'{RAW_DATA_DIR}/**/{table}', recursive=True):
        try:
            chunk = pd.read_csv(
                f, usecols=['PATIENT', cols['date_col'],
                             cols['code_col'], cols['desc_col']],
                low_memory=False)
            chunk[cols['code_col']] = pd.to_numeric(
                chunk[cols['code_col']], errors='coerce')
            rows = chunk[chunk[cols['code_col']].isin(T2D_ALL_CODES)].copy()
            if not rows.empty:
                rows = rows.rename(columns={
                    cols['date_col']: 'Td',
                    cols['code_col']: 'Diagnosis_Code',
                    cols['desc_col']: 'Diagnosis_Description',
                })
                rows['Diagnosis_Source'] = table
                td_chunks.append(rows[['PATIENT','Td','Diagnosis_Code',
                                        'Diagnosis_Description','Diagnosis_Source']])
        except Exception:
            pass

df_td_all = (pd.concat(td_chunks, ignore_index=True)
               .assign(Td=lambda x: pd.to_datetime(x['Td'], errors='coerce'))
               .dropna(subset=['Td']))

# Earliest diagnosis per patient
df_td = (df_td_all.sort_values('Td')
                  .drop_duplicates(subset='PATIENT', keep='first')
                  .rename(columns={'Td': 'Date_Diagnosis_Td'}))

N_ANY = len(df_td)
print(f"  {N_ANY:,} patients with any T2D-related code")

# ── STEP 3 : Validate as Type 2 — the critical filter ────────────────────────
# A patient is confirmed T2D if they have at least one T2D-SPECIFIC code.
# Patients who ONLY ever have the generic 44054006 are removed because
# that code is shared with Type 1 and gestational diabetes.
print("\nSTEP 3 — Validating as Type 2 specifically...")
print("  Logic: patient must have ≥1 T2D-specific complication code,")
print("         not just the generic 'Diabetes' code 44054006.")

confirmed_t2d_ids = set()
for f in glob.glob(f'{RAW_DATA_DIR}/**/conditions.csv', recursive=True):
    try:
        chunk = pd.read_csv(f, usecols=['PATIENT','CODE'], low_memory=False)
        chunk['CODE'] = pd.to_numeric(chunk['CODE'], errors='coerce')
        confirmed_t2d_ids.update(
            chunk[chunk['CODE'].isin(T2D_SPECIFIC_CODES)]['PATIENT'].unique())
    except Exception:
        pass

df_td = df_td[df_td['PATIENT'].isin(confirmed_t2d_ids)].copy()
N_VALIDATED = len(df_td)
print(f"  {N_VALIDATED:,} confirmed T2D patients  "
      f"(removed {N_ANY - N_VALIDATED:,} generic-only 'Diabetes' cases)")

# ── STEP 4 : First treatment date (T0) + drug class ──────────────────────────
print("\nSTEP 4 — Finding first treatment date (T0)...")
t0_chunks = []
for f in glob.glob(f'{RAW_DATA_DIR}/**/medications.csv', recursive=True):
    try:
        chunk = pd.read_csv(f, usecols=['PATIENT','CODE','START'], low_memory=False)
        chunk['CODE'] = pd.to_numeric(chunk['CODE'], errors='coerce')
        rows = chunk[chunk['CODE'].isin(T2D_MED_CODES.keys())].copy()
        if not rows.empty:
            t0_chunks.append(rows)
    except Exception:
        pass

df_t0 = (pd.concat(t0_chunks, ignore_index=True)
           .assign(T0=lambda x: pd.to_datetime(x['START'], errors='coerce'))
           .dropna(subset=['T0'])
           .sort_values('T0')
           .drop_duplicates(subset='PATIENT', keep='first')
           .assign(Drug_Class=lambda x: x['CODE'].map(T2D_MED_CODES))
           .rename(columns={'T0': 'Date_Treatment_T0'})
           [['PATIENT','Date_Treatment_T0','Drug_Class']])

N_TREATED = df_td['PATIENT'].isin(df_t0['PATIENT']).sum()
print(f"  {N_TREATED:,} validated T2D patients have ≥1 treatment record")

# ── STEP 5 : Global start date ────────────────────────────────────────────────
print("\nSTEP 5 — Computing Global_Start_Date per patient...")
DATE_TABLES = {
    'conditions.csv':'START', 'medications.csv':'START',
    'encounters.csv':'DATE',  'careplans.csv':  'START',
    'procedures.csv':'DATE',  'observations.csv':'DATE',
}
min_chunks = []
for table, date_col in DATE_TABLES.items():
    for f in glob.glob(f'{RAW_DATA_DIR}/**/{table}', recursive=True):
        try:
            chunk = pd.read_csv(f, usecols=['PATIENT', date_col], low_memory=False)
            chunk[date_col] = pd.to_datetime(chunk[date_col], errors='coerce')
            min_chunks.append(chunk.groupby('PATIENT')[date_col].min().rename('date'))
        except Exception:
            pass

df_global_start = (pd.concat(min_chunks)
                     .groupby('PATIENT').min()
                     .reset_index()
                     .rename(columns={'date': 'Global_Start_Date'}))
print(f"  Global start dates found for {len(df_global_start):,} patients")

# ── STEP 6 : Assemble ─────────────────────────────────────────────────────────
print("\nSTEP 6 — Assembling master cohort...")
cohort = (df_td
          .merge(df_t0,           on='PATIENT', how='left')
          .merge(df_patients,     on='PATIENT', how='inner')
          .merge(df_global_start, on='PATIENT', how='left'))

cohort['Observation_End'] = cohort['DEATHDATE'].fillna(STUDY_END)
cohort['Is_Dead']         = cohort['DEATHDATE'].notna()
cohort['Total_Obs_Years'] = ((cohort['Observation_End'] - cohort['Global_Start_Date'])
                              .dt.days / 365.25).round(1)

# Ages
cohort['Age_at_Diagnosis'] = ((cohort['Date_Diagnosis_Td'] - cohort['BIRTHDATE'])
                               .dt.days / 365.25).round(1)
cohort['Age_at_Treatment'] = ((cohort['Date_Treatment_T0'] - cohort['BIRTHDATE'])
                               .dt.days / 365.25).round(1)
cohort['Age_at_Death']     = np.where(
    cohort['Is_Dead'],
    ((cohort['DEATHDATE'] - cohort['BIRTHDATE']).dt.days / 365.25).round(1),
    np.nan)

# Index dates
cohort['Date_T0_plus5']   = cohort['Date_Treatment_T0'] + pd.DateOffset(years=5)
cohort['Date_T0_plus10']  = cohort['Date_Treatment_T0'] + pd.DateOffset(years=10)
cohort['Age_at_T0_plus5'] = ((cohort['Date_T0_plus5'] - cohort['BIRTHDATE'])
                              .dt.days / 365.25).round(1)
cohort['Age_at_T0_plus10']= ((cohort['Date_T0_plus10'] - cohort['BIRTHDATE'])
                              .dt.days / 365.25).round(1)

# Follow-up per index date (capped at 5 years)
def fu_years(df, index_col):
    idx    = df[index_col]
    fu_end = pd.concat([df['Observation_End'],
                        idx + pd.DateOffset(years=5)], axis=1).min(axis=1)
    return ((fu_end - idx).dt.days / 365.25).round(1)

cohort['FU_Years_Td']       = fu_years(cohort, 'Date_Diagnosis_Td')
cohort['FU_Years_T0']       = fu_years(cohort, 'Date_Treatment_T0')
cohort['FU_Years_T0plus5']  = np.where(
    cohort['Observation_End'] >= cohort['Date_T0_plus5'],
    fu_years(cohort, 'Date_T0_plus5'), np.nan)
cohort['FU_Years_T0plus10'] = np.where(
    cohort['Observation_End'] >= cohort['Date_T0_plus10'],
    fu_years(cohort, 'Date_T0_plus10'), np.nan)

# ── STEP 7 : Attrition summary ────────────────────────────────────────────────
def is_eligible(df, index_col):
    idx         = df[index_col]
    lookback_ok = (idx - df['BIRTHDATE']).dt.days / 365.25 >= 5
    followup_ok = (
        ((df['Observation_End'] - idx).dt.days / 365.25 >= 5) |
        (df['Is_Dead'] & (df['Observation_End'] >= idx))
    )
    return lookback_ok & followup_ok

cohort_with_t0 = cohort.dropna(subset=['Date_Treatment_T0']).copy()

print("\n" + "=" * 65)
print("ATTRITION FLOWCHART")
print("=" * 65)
print(f"  Full dataset patients                     : {len(df_patients):>8,}")
print(f"  T2D any code                              : {N_ANY:>8,}")
print(f"  Confirmed Type 2 (specific code required) : {N_VALIDATED:>8,}")
print(f"  Confirmed T2D + treatment record          : {N_TREATED:>8,}")

print(f"\n  {'Index':<14} {'N eligible':>10}   "
      f"{'Earliest index':>15}   {'Latest index':>12}   {'Median FU':>10}")
print("  " + "-" * 68)

for label, df_sub, index_col in [
    ('Td',       cohort,          'Date_Diagnosis_Td'),
    ('T0',       cohort_with_t0,  'Date_Treatment_T0'),
    ('T0+5',     cohort_with_t0,  'Date_T0_plus5'),
    ('T0+10',    cohort_with_t0,  'Date_T0_plus10'),
]:
    elig    = is_eligible(df_sub, index_col)
    sub     = df_sub[elig]
    n       = len(sub)
    idx     = sub[index_col]
    fu_end  = pd.concat([sub['Observation_End'],
                         idx + pd.DateOffset(years=5)], axis=1).min(axis=1)
    med_fu  = f"{(fu_end - idx).dt.days.median() / 365.25:.1f} yrs"
    print(f"  {label:<14} {n:>10,}   "
          f"{str(idx.min().date()):>15}   "
          f"{str(idx.max().date()):>12}   {med_fu:>10}")

print("=" * 65)

# ── Final column order and save ───────────────────────────────────────────────
final_cols = [
    'PATIENT','GENDER','RACE','ETHNICITY',
    'BIRTHDATE','DEATHDATE','Is_Dead','Age_at_Death',
    'Global_Start_Date','Observation_End','Total_Obs_Years',
    'Date_Diagnosis_Td','Age_at_Diagnosis',
    'Diagnosis_Code','Diagnosis_Description','Diagnosis_Source',
    'Date_Treatment_T0','Age_at_Treatment','Drug_Class',
    'Date_T0_plus5','Age_at_T0_plus5',
    'Date_T0_plus10','Age_at_T0_plus10',
    'FU_Years_Td','FU_Years_T0','FU_Years_T0plus5','FU_Years_T0plus10',
]
cohort = cohort[[c for c in final_cols if c in cohort.columns]]
cohort.to_csv('/content/cohort_master.csv', index=False)
print(f"\n✅ Saved /content/cohort_master.csv  —  "
      f"{len(cohort):,} patients × {len(cohort.columns)} columns")
print("\nSample (3 rows):")
print(cohort.head(3).to_string())

In [ ]:
=================================================================
COHORT BUILDING
=================================================================

STEP 1 — Loading patient demographics...
  1,193,612 patients loaded

STEP 2 — Finding T2D diagnosis dates (Td)...
  86,604 patients with any T2D-related code

STEP 3 — Validating as Type 2 specifically...
  Logic: patient must have ≥1 T2D-specific complication code,
         not just the generic 'Diabetes' code 44054006.
  62,204 confirmed T2D patients  (removed 24,400 generic-only 'Diabetes' cases)

STEP 4 — Finding first treatment date (T0)...
  60,047 validated T2D patients have ≥1 treatment record

STEP 5 — Computing Global_Start_Date per patient...
  Global start dates found for 1,424,253 patients

STEP 6 — Assembling master cohort...

=================================================================
ATTRITION FLOWCHART
=================================================================
  Full dataset patients                     : 1,193,612
  T2D any code                              :   86,604
  Confirmed Type 2 (specific code required) :   62,204
  Confirmed T2D + treatment record          :   60,047

  Index          N eligible    Earliest index   Latest index    Median FU
  --------------------------------------------------------------------
  Td                 45,069        1927-10-23     2016-07-13      5.0 yrs
  T0                 42,504        1931-08-28     2017-04-27      5.0 yrs
  T0+5               38,169        1936-08-28     2016-09-06      5.0 yrs
  T0+10              32,130        1941-08-28     2017-02-10      5.0 yrs
=================================================================

✅ Saved /content/cohort_master.csv  —  46,330 patients × 27 columns

Sample (3 rows):
                                PATIENT GENDER   RACE ETHNICITY  BIRTHDATE  DEATHDATE  Is_Dead  Age_at_Death Global_Start_Date Observation_End  Total_Obs_Years Date_Diagnosis_Td  Age_at_Diagnosis  Diagnosis_Code Diagnosis_Description Diagnosis_Source Date_Treatment_T0  Age_at_Treatment Drug_Class Date_T0_plus5  Age_at_T0_plus5 Date_T0_plus10  Age_at_T0_plus10  FU_Years_Td  FU_Years_T0  FU_Years_T0plus5  FU_Years_T0plus10
0  e096ef16-af8e-43e5-b2dc-108b05563f7d      M  asian   chinese 1908-07-11 1972-10-07     True          64.2        1927-10-23      1972-10-07             45.0        1927-10-23              19.3      44054006.0              Diabetes   conditions.csv        1945-10-21              37.3  Metformin    1950-10-21             42.3     1955-10-21              47.3          5.0          5.0               5.0                5.0
1  0f3c861e-a7a8-44f3-a386-8ec8df147d3e      F  white     irish 1909-12-24 1968-11-17     True          58.9        1912-11-30      1968-11-17             56.0        1928-05-04              18.4      44054006.0              Diabetes   conditions.csv        1944-08-01              34.6  Metformin    1949-08-01             39.6     1954-08-01              44.6          5.0          5.0               5.0                5.0
2  6808a8d8-4d3b-42e4-86f1-3238226cb906      M  white   swedish 1907-09-27 1986-02-21     True          78.4        1912-12-07      1986-02-21             73.2        1928-08-06              20.9      44054006.0              Diabetes   conditions.csv        1939-04-07              31.5  Metformin    1944-04-07             36.5     1949-04-07              41.5          5.0          5.0               5.0                5.0


In [ ]:
# =============================================================================
# CELL 2: FILTERED STUDY DATA (only cohort patients, patients.csv address-safe)
# Purpose : Copy all clinical CSV shards filtered to cohort patients only.
#           patients.csv uses usecols to skip the ADDRESS column.
# Input   : /content/Final_Thesis_Data   /content/cohort_master.csv
# Output  : /content/Filtered_Study_Data/  (mirrored folder structure)
#           /content/Filtered_Study_Data.zip
# Packages: pandas, glob, os, shutil
# =============================================================================

import os, glob, shutil
import pandas as pd

RAW_DATA_DIR  = '/content/Final_Thesis_Data'
FILTERED_DIR  = '/content/Filtered_Study_Data'

PATIENT_COLS  = [          # ADDRESS excluded — contains unquoted commas
    'ID','BIRTHDATE','DEATHDATE','SSN','DRIVERS','PASSPORT',
    'PREFIX','FIRST','LAST','SUFFIX','MAIDEN','MARITAL',
    'RACE','ETHNICITY','GENDER','BIRTHPLACE'
]

cohort_ids = set(pd.read_csv('/content/cohort_master.csv', usecols=['PATIENT'])['PATIENT'])
print(f"Filtering all shards to {len(cohort_ids):,} cohort patients...\n")

for src_path in sorted(glob.glob(f'{RAW_DATA_DIR}/**/*.csv', recursive=True)):
    rel_path = os.path.relpath(src_path, RAW_DATA_DIR)
    dst_path = os.path.join(FILTERED_DIR, rel_path)
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)

    table_name = os.path.basename(src_path)

    try:
        if table_name == 'patients.csv':
            df = pd.read_csv(src_path, usecols=PATIENT_COLS, low_memory=False)
            df = df[df['ID'].isin(cohort_ids)]
        else:
            df = pd.read_csv(src_path, low_memory=False)
            if 'PATIENT' in df.columns:
                df = df[df['PATIENT'].isin(cohort_ids)]

        if not df.empty:
            df.to_csv(dst_path, index=False)
            print(f"  ✅  {rel_path:<55}  {len(df):>7,} rows")

    except Exception as e:
        print(f"  ⚠️  {rel_path}  →  {e}")

shutil.make_archive('/content/Filtered_Study_Data', 'zip', FILTERED_DIR)
print("\n✅ Done — Filtered_Study_Data.zip ready to download")